# Repeating earthquakes at SAFOD, and what they say about creep

Every number below is read from a file produced by a tracked script. Nothing is
typed in by hand, and each cell names the script that made its input.

## The question

A *repeating earthquake* is the same patch of fault rupturing more than once. The
patch is locked; the fault around it creeps aseismically and reloads it until it
fails again. So **recurrence interval + magnitude gives the creep rate of the
surrounding fault**, at depths no instrument reaches directly
(Nadeau & McEvilly 1999, *Science* **285**:718).

The hard part is not finding similar earthquakes. It is separating

- **repeaters** — one patch, rupturing repeatedly, from
- **neighbours** — adjacent patches that look almost identical from 3 km away.

Only the first gives a recurrence interval. Bill Ellsworth put it exactly this way:

> *"you will want to find events that are close in magnitude and location, **which
> will then need to be verified as either repeaters or neighbors**."*

## How to reproduce

```bash
ml gcc/12.4.0 && conda activate das
cd notebooks/faultzone/repeaters
sbatch phaseA_job.sh     # coverage audit      -> phaseA_events.csv
sbatch beta_job.sh       # similarity beta     -> beta_similarity.csv
sbatch seq_job.sh        # clusters -> creep   -> sequences_creep.csv
```

Companion documents: `REPEATER_PLAN.md` (the procedure and its sources),
`METHODS_STATUS.md` (methods, literature, and the error log).

In [ ]:
import os, collections
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 170)
for f in ['phaseA_events.csv','beta_similarity.csv','sequences_creep.csv']:
    print(f'{"OK " if os.path.exists(f) else "MISSING"} {f}')

## 1. The catalog, and which events the DAS actually recorded

Ellsworth supplied the starting catalog: NCEDC **double-difference** relocations,
2024-05-01 to 2026-04-01, within 15 km of SAFOD — **329 events**. Double-difference
matters because routine locations are not precise enough to say anything about
whether two events share a patch.

Coverage is audited at *file-interval* resolution against **both** manifests. An
earlier audit used one manifest and a later one used day-level coverage; both were
wrong, in opposite directions.

*Script:* `phaseA_coverage.py`

In [ ]:
E = pd.read_csv('phaseA_events.csv')
E['time'] = pd.to_datetime(E.time, utc=True, format='mixed')
print(f'DDRT catalog          : {len(E)} events   (Ellsworth reports 329)')
print(f'recorded by the DAS   : {int(E.cov_full.sum())}')
print(f'not recorded          : {int((~E.cov_full).sum())}')
print(f'pairs to test         : {int(E.cov_full.sum())*(int(E.cov_full.sum())-1)//2:,}')
print()
print(E[E.cov_full].groupby(E.time.dt.to_period("M")).size()
      .to_frame('events recorded').T.to_string())

## 2. Similarity — Nadeau's β

> *"The similarity measure, β, ... is based on a network-wide characterization of
> maximum cross-correlation coefficient values for **P and S waves** between pairs
> of earthquakes"* — Nadeau, Foxall & McEvilly 1995, *Science* **267**:503

Measured here on **HRSN** (network BP), the borehole array the Parkfield repeater
catalogs were built on, using three components, P and S windows separately, and a
per-station alignment before windowing.

**Why alignment matters.** Catalog origin times carry 0.1–0.5 s of error. Correlating
at zero lag drives *identical* waveforms to zero; searching ±1 s inside a 1.5 s
window instead maximises over ~30 lags of a ~45-DOF correlation and inflates the
null to ~0.4. Both mistakes were made here before being caught. The fix is one bulk
alignment per station, then a ±0.1 s residual search.

*Script:* `beta_similarity.py`

In [ ]:
B = pd.read_csv('beta_similarity.csv')
for c in ('t_i','t_j'): B[c] = pd.to_datetime(B[c], utc=True, format='mixed')
B = B[B.days > 1]                      # drop same-instant catalog double-listings
print(f'{len(B):,} pairs measured\n')
print(f'  median beta (the null) : {B.beta.median():.3f}')
print(f'  99th percentile        : {B.beta.quantile(0.99):.3f}')
print(f'  maximum                : {B.beta.max():.4f}')
for t in [0.6,0.7,0.8,0.9,0.95,0.98]:
    print(f'  beta > {t:.2f} : {int((B.beta>t).sum()):5d} pairs ({100*(B.beta>t).mean():.3f}%)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].hist(B.beta, bins=np.arange(0,1.02,0.02), color='0.6')
ax[0].set(yscale='log', xlabel=r'similarity $\beta$', ylabel='pairs',
          title='A  Two populations: background, then a flat tail')
ax[0].axvspan(0.6, 0.9, color='C1', alpha=.15)
ax[0].text(0.62, ax[0].get_ylim()[1]*0.3, "Nadeau's gap\n0.6-0.9", fontsize=8)
ax[1].scatter(B.days, B.beta, s=4, c='0.7')
hi = B[B.beta >= 0.90]
ax[1].scatter(hi.days, hi.beta, s=28, c='C3', label=r'$\beta \geq 0.90$')
ax[1].set(xlabel='days between events', ylabel=r'$\beta$',
          title='B  High-similarity pairs span the whole record')
ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### The threshold: use Nadeau's criterion, not his number

Nadeau chose β ≥ 0.98 as *"a value at which the defined cluster population changed
little with β"* — a **stability** criterion.

Our β tops out at **0.970**, with zero pairs at 0.98. That is not evidence Parkfield
stopped producing repeaters: they found **63% of 1700 events** in clusters, and a
fall to literally zero is not credible. The absolute scale differs by ~0.02 —
different band, window, aggregation, and 1987-era instruments.

What *does* match is the population size. Their 294 clusters from 1700 events implies
roughly 0.1% of pairs above threshold; we have 0.06% above 0.90. Same order.

**Honest caveat:** the cluster count declines monotonically in our data (20, 20, 17,
16, 13, 10, 7, 5, 2, 0), so there is no clean plateau to point at. β ≥ 0.90 is chosen
as the scale-shifted equivalent of their 0.98, and that choice is a judgement, not a
measurement.

In [ ]:
def clusters_at(thresh):
    sel = B[B.beta >= thresh]; par = {}
    def find(x):
        par.setdefault(x, x)
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for a, b in zip(sel.i, sel.j):
        ra, rb = find(a), find(b)
        if ra != rb: par[ra] = rb
    g = collections.defaultdict(list)
    for x in list(par): g[find(x)].append(x)
    return [sorted(v) for v in g.values() if len(v) > 1]

print(f'{"beta":>7}{"pairs":>8}{"clusters":>10}{"events":>8}')
for t in [0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.94,0.96,0.98]:
    cl = clusters_at(t)
    print(f'{t:7.2f}{int((B.beta>=t).sum()):8d}{len(cl):10d}{sum(len(c) for c in cl):8d}')

## 3. Clusters

Events are linked into clusters by Nadeau's equivalence-class algorithm: any two
events sharing a supra-threshold pair belong to the same cluster.

**A cluster is not yet a sequence.** Nadeau 1995 found that *"for clusters of three
or more events, 80 to 90% were complex"* — their members split into subgroups, each
a different patch, once you look at high-frequency detail. That subdivision is the
repeaters-versus-neighbours step and is **not** implemented here; with mostly
two-event clusters it does not bite yet, but it would with a larger catalog.

In [ ]:
ev = pd.read_csv('phaseA_events.csv')
ev['time'] = pd.to_datetime(ev.time, utc=True, format='mixed')
ev = ev[ev.cov_full].reset_index(drop=True)
cl = sorted(clusters_at(0.90))
print(f'{len(cl)} clusters at beta >= 0.90\n')
for k, mem in enumerate(cl):
    rows = sorted((str(ev.time.iloc[m])[:10], float(ev.mag.iloc[m])) for m in mem)
    bmax = B[(B.i.isin(mem)) & (B.j.isin(mem))].beta.max()
    print(f'  {k}: beta {bmax:.3f}   ' + '   '.join(f'{d} M{m:.2f}' for d, m in rows))

## 4. Confirmation and creep

Nadeau & Johnson 1998 confirm a sequence by *"virtual collocation of the events,
their quasi-periodic recurrence, and their nearly identical magnitudes."*

**Collocation cannot be tested here**, and that is worth stating plainly. Nadeau 1995
notes that *routine* locations scatter genuine repeaters over up to 200 m, while
waveform-based relative relocation puts them within 10–20 m. Our DDRT separations are
167–771 m — exactly the routine-catalog scatter — so they carry no information about
whether two events share a patch. Location is reported, never used as a gate.

The gates actually applied:

| gate | value | source |
|---|---|---|
| nearly identical magnitudes | ΔM ≤ 0.2 | Nadeau & Johnson 1998 |
| quasi-periodic recurrence | CV ≤ 0.5 | Nadeau & Johnson 1998 |
| independent loading cycles | interval ≥ 30 d | Waldhauser & Ellsworth 2002 |
| physically plausible | rate < 50 mm/yr | the fault's own slip rate |

The 30-day rule matters: without it, bursts of events days apart produced creep rates
up to 866,000 mm/yr.

*Script:* `sequences_and_creep.py`

In [ ]:
R = pd.read_csv('sequences_creep.csv')
show = ['cluster','n','mean_interval_d','dmag','mag_mean','radius_m',
        'rate_crack_mmyr','rate_nj_mmyr','passes']
print(R[show].to_string(index=False))
print(f'\n{int(R.passes.sum())} of {len(R)} sequences pass every gate')

In [ ]:
P = R[R.passes]
fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(len(P))
ax.bar(x-0.2, P.rate_crack_mmyr, 0.4, label='circular crack (3 MPa)', color='0.6')
ax.bar(x+0.2, P.rate_nj_mmyr, 0.4, label='Nadeau & Johnson 1998', color='C0')
ax.axhspan(25, 30, color='C2', alpha=.25, label='Parkfield creep, geodetic')
ax.set_xticks(x); ax.set_xticklabels([f'seq {int(c)}' for c in P.cluster])
ax.set(ylabel='creep rate (mm/yr)', yscale='log',
       title='Two slip models, same recurrence intervals')
ax.legend(fontsize=8); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.show()

## 5. Result

**Two sequences survive every published gate.**

| sequence | events | interval | ΔM | source radius | creep (N&J) |
|---|---|---|---|---|---|
| 0 | 2024-05-13 M0.78 → 2025-07-27 M0.84 | 440 d | 0.06 | 14.4 m | **31.8 mm/yr** |
| 3 | 2024-07-08 M0.65 → 2025-04-06 M0.81 | 272 d | 0.16 | 13.2 m | **49.1 mm/yr** |

Those rates bracket the geodetic creep rate of the San Andreas near Parkfield
(~25–30 mm/yr). The circular-crack model gives 0.9–1.3 mm/yr, roughly 25× too low —
the known underestimate for small repeaters that motivated Nadeau & Johnson's
empirical relation in the first place.

### What this is not

**The agreement is partly circular.** Nadeau & Johnson calibrated their scaling
*against geodetic creep at Parkfield*. Reproducing ~30 mm/yr there is a consistency
check, not an independent measurement. The honest claim is that these sequences behave
like the Parkfield repeaters the relation was built on.

**Two sequences, two intervals.** No recurrence *series*, so no CV, no time
dependence, and no useful error bar. Nadeau & McEvilly had 844 intervals from 11
years; this is 14 months.

**Neighbours are not yet excluded.** The high-frequency subdivision that separates
patches within a cluster is not implemented. With mostly two-event clusters it
changes nothing here, but it is the step that would matter with more data.

**β ≥ 0.90 is a judgement.** The scale offset from Nadeau's 0.98 is argued from
population size, not measured, and our cluster count shows no plateau.

### What would move it forward

1. **Template matching** on the continuous archive — the intervals of 440 and 272
   days are suspiciously long for M0.7 repeaters, and events below catalog
   completeness are the obvious explanation. This is the single highest-value next
   step.
2. **Waveform-based relative relocation**, which would make the collocation test
   usable and enable high-frequency subdivision.
3. **Historical catalogs** (Waldhauser–Schaff, 1984–2019) — a sequence active then
   and now inherits decades of recurrence rather than one interval.